# EP1 — Datos y baseline E0

Este notebook implementa la primera etapa: validación de Fashion-MNIST, preprocesamiento reproducible y baseline E0. Las decisiones de arquitectura se compararán después; el conjunto oficial de test permanece sellado.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))

from ep1_fashion_mnist.data import CLASS_NAMES, prepare_fashion_mnist
from ep1_fashion_mnist.experiment import run_validation_experiment
from ep1_fashion_mnist.visualization import plot_class_distribution, plot_sample_grid

print('Python executable:', sys.executable)
print('Project root:', ROOT)

Python executable: C:\Users\yvl\Documents\GitHub\ep1-deep-learning-fashion-mnist\.venv\Scripts\python.exe
Project root: C:\Users\yvl\Documents\GitHub\ep1-deep-learning-fashion-mnist


## Contrato y preprocesamiento

Se carga Fashion-MNIST mediante Keras. Se validan forma, tipo, rango y balance de los splits oficiales. Desde el train oficial se generan índices estratificados 90/10 con seed 42; luego las imágenes se convierten a `float32` en `[0, 1]` y las etiquetas se codifican one-hot. La imagen se conserva como matriz 28×28 para el análisis visual; `Flatten` vive dentro de la MLP.

In [2]:
data = prepare_fashion_mnist()
print(data.train_summary)
print(data.test_summary)
print(data.partition_summary)
print('Processed:', data.X_train.shape, data.X_val.shape, data.X_test.shape, data.X_train.dtype)
print('Classes:', CLASS_NAMES)

SplitSummary(name='official_train', count=60000, image_shape=(28, 28), image_dtype='uint8', label_dtype='uint8', pixel_range=(0, 255), class_counts={0: 6000, 1: 6000, 2: 6000, 3: 6000, 4: 6000, 5: 6000, 6: 6000, 7: 6000, 8: 6000, 9: 6000})
SplitSummary(name='official_test', count=10000, image_shape=(28, 28), image_dtype='uint8', label_dtype='uint8', pixel_range=(0, 255), class_counts={0: 1000, 1: 1000, 2: 1000, 3: 1000, 4: 1000, 5: 1000, 6: 1000, 7: 1000, 8: 1000, 9: 1000})
PartitionSummary(source_count=60000, train_count=54000, validation_count=6000)
Processed: (54000, 28, 28) (6000, 28, 28) (10000, 28, 28) float32
Classes: ('T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot')


## Exploración inicial

Las clases del train resultante permanecen balanceadas (5.400 ejemplos por clase). La cuadrícula muestra la resolución limitada y semejanzas visuales entre algunas prendas; más adelante esto permitirá interpretar confusiones sin convertir una observación visual en una conclusión sobre desempeño.

In [3]:
figures = ROOT / 'results' / 'figures'
plot_sample_grid(data.X_train, data.y_train, CLASS_NAMES, output_path=figures / 'D0_train_examples.png')
plot_class_distribution(data.y_train, CLASS_NAMES, output_path=figures / 'D0_train_class_distribution.png')
print('Saved EDA figures in', figures)

Saved EDA figures in C:\Users\yvl\Documents\GitHub\ep1-deep-learning-fashion-mnist\results\figures


## Baseline E0

E0 usa `[256, 128]`, ReLU, Softmax, categorical crossentropy, SGD con learning rate 0,01, batch 128 y 20 épocas, sin regularización. Es una hipótesis de referencia, no el modelo final. Entrenamos y medimos exclusivamente sobre validation.

In [4]:
config_path = ROOT / 'configs' / 'baseline.json'
print(json.loads(config_path.read_text(encoding='utf-8')))
metrics = run_validation_experiment(config_path, output_directory=ROOT / 'results' / 'runs' / 'e0')
print(metrics)

{'experiment_id': 'E0', 'status': 'validated_on_validation', 'seed': 42, 'dataset': 'fashion_mnist', 'validation_fraction': 0.1, 'stratified_split': True, 'input_shape': [28, 28], 'hidden_layers': [256, 128], 'hidden_activation': 'relu', 'output_units': 10, 'output_activation': 'softmax', 'label_encoding': 'one_hot', 'loss': 'categorical_crossentropy', 'optimizer': 'sgd', 'learning_rate': 0.01, 'batch_size': 128, 'epochs': 20, 'dropout': 0.0, 'batch_normalization': False, 'l2_strength': 0.0, 'early_stopping': False}


Epoch 1/20


422/422 - 2s - 4ms/step - accuracy: 0.6699 - loss: 1.1065 - val_accuracy: 0.7592 - val_loss: 0.7395


Epoch 2/20


422/422 - 1s - 2ms/step - accuracy: 0.7755 - loss: 0.6666 - val_accuracy: 0.8027 - val_loss: 0.6030


...

Epoch 19/20


422/422 - 1s - 3ms/step - accuracy: 0.8675 - loss: 0.3789 - val_accuracy: 0.8713 - val_loss: 0.3696


Epoch 20/20


422/422 - 1s - 3ms/step - accuracy: 0.8687 - loss: 0.3751 - val_accuracy: 0.8722 - val_loss: 0.3666


ValidationMetrics(accuracy=0.8721666666666666, precision_weighted=0.873749979147054, recall_weighted=0.8721666666666666, f1_weighted=0.872619019330949)


## Lectura inicial

En la ejecución registrada, E0 alcanzó accuracy de validation 0,8722 y F1 ponderado 0,8726. Las curvas y la tabla seleccionada se guardan con ID E0. Este resultado establece un control para las comparaciones posteriores; no justifica aún escoger hiperparámetros ni consultar test.